# Melanoma Research Assistant — Drug Data Extension (PubChem + ChEMBL) & Memory-Enabled RAG

This notebook builds the drug knowledge base dynamically and then a combined RAG chain that:

1. **Discovers** melanoma-relevant drugs directly from **ChEMBL** (indication + target search)
2. Fetches **PubChem** compound description and **ChEMBL** mechanism/target/indication data for each
3. Combines both into one record per drug, embeds it, and stores it in a separate Pinecone **`drugs`
   namespace**
4. Adds a lightweight **router** that decides, per question, whether to search literature, drugs, or both
5. Adds **conversational memory** so follow-up questions work

> Requires `OPENAI_API_KEY` and `PINECONE_API_KEY` env vars, and internet access to
> `pubchem.ncbi.nlm.nih.gov` and `www.ebi.ac.uk` (ChEMBL).


## 0. Install dependencies

In [ ]:
# !pip install -q requests langchain langchain-core langchain-text-splitters langchain-openai langchain-pinecone pinecone pandas tqdm python-dotenv

## 1. Configuration

We reuse the **same Pinecone index** created in notebook 1. Literature chunks were upserted into the
default namespace (`""`); drug chunks go into a separate `"drugs"` namespace so the two data types
never collide and can be queried independently or together.

`MAX_DRUGS` caps how many discovered drugs get ingested.


In [ ]:
import os
import time
import requests
import pandas as pd

from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]

PINECONE_INDEX_NAME = "melanoma-kb"
LITERATURE_NAMESPACE = ""       # matches notebook 1 (default namespace)
DRUGS_NAMESPACE = "drugs"
EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-4o-mini"
TOP_K = 5

MAX_DRUGS = 100  # cap on how many discovered drugs get ingested

# Indication terms used to find melanoma-relevant drugs via ChEMBL's drug_indication endpoint.
MELANOMA_INDICATION_TERMS = ["melanoma"]

# Target gene symbols used to find melanoma-relevant drugs via ChEMBL's mechanism/target endpoints,
# as a second discovery path (catches drugs indication data alone might miss).
MELANOMA_TARGET_SYMBOLS = ["BRAF", "MAP2K1", "MAP2K2", "PDCD1", "CD274", "CTLA4", "KIT"]

PUBCHEM_BASE = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
CHEMBL_BASE = "https://www.ebi.ac.uk/chembl/api/data"


## 2. Discover melanoma-relevant drugs from ChEMBL

Two discovery paths, combined and de-duplicated by `molecule_chembl_id`:

1. **By indication** — `drug_indication` records where the EFO/MedDRA term contains "melanoma"
2. **By target** — known melanoma-relevant targets (BRAF, MEK1/2, PD-1, PD-L1, CTLA-4, KIT) resolved
   to ChEMBL target IDs, then drugs with a mechanism against those targets

Each ChEMBL ID is then resolved to a preferred drug name (and clinical phase) via the `molecule`
endpoint, batched to keep request counts low.


In [2]:
def fetch_all_pages(url: str, key: str, max_items: int = 500) -> list[dict]:
    """Follow ChEMBL's pagination ('page_meta.next') until max_items or no more pages."""
    items = []
    while url and len(items) < max_items:
        try:
            resp = requests.get(url, timeout=20)
            resp.raise_for_status()
        except Exception as e:
            print(f"[ChEMBL] Pagination error at {url}: {e}")
            break
        data = resp.json()
        items.extend(data.get(key, []))
        next_path = data.get("page_meta", {}).get("next")
        url = f"https://www.ebi.ac.uk{next_path}" if next_path else None
        time.sleep(0.3)
    return items[:max_items]


def discover_chembl_ids_by_indication(terms: list[str], max_items: int = 300) -> dict[str, str]:
    """Return {chembl_id: matched_efo_term} for drugs indicated for any of the given terms."""
    hits = {}
    for term in terms:
        url = f"{CHEMBL_BASE}/drug_indication.json?efo_term__icontains={term}&limit=100"
        for record in fetch_all_pages(url, "drug_indications", max_items=max_items):
            cid = record.get("molecule_chembl_id")
            if cid:
                hits[cid] = record.get("efo_term")
    print(f"Indication search: {len(hits)} candidate ChEMBL IDs")
    return hits


def resolve_target_chembl_ids(gene_symbols: list[str]) -> list[str]:
    """Resolve gene symbols (e.g. 'BRAF') to ChEMBL target IDs via target search."""
    target_ids = []
    for symbol in gene_symbols:
        url = f"{CHEMBL_BASE}/target/search?q={symbol}&format=json"
        try:
            resp = requests.get(url, timeout=20)
            resp.raise_for_status()
            targets = resp.json().get("targets", [])
            # keep single-protein human targets only, to avoid noisy multi-target complexes
            human_targets = [t for t in targets if t.get("target_type") == "SINGLE PROTEIN"
                              and t.get("organism") == "Homo sapiens"]
            target_ids.extend(t["target_chembl_id"] for t in human_targets[:2])
        except Exception as e:
            print(f"[ChEMBL] Target search error for '{symbol}': {e}")
        time.sleep(0.3)
    return list(set(target_ids))


def discover_chembl_ids_by_target(gene_symbols: list[str], max_items: int = 300) -> dict[str, str]:
    """Return {chembl_id: matched_target_symbol} for drugs with a mechanism against these targets."""
    target_ids = resolve_target_chembl_ids(gene_symbols)
    hits = {}
    for target_id in target_ids:
        url = f"{CHEMBL_BASE}/mechanism.json?target_chembl_id={target_id}&limit=100"
        for record in fetch_all_pages(url, "mechanisms", max_items=max_items):
            cid = record.get("molecule_chembl_id")
            if cid:
                hits[cid] = target_id
    print(f"Target search: {len(hits)} candidate ChEMBL IDs")
    return hits


indication_hits = discover_chembl_ids_by_indication(MELANOMA_INDICATION_TERMS)
target_hits = discover_chembl_ids_by_target(MELANOMA_TARGET_SYMBOLS)

all_candidate_ids = list(set(indication_hits) | set(target_hits))
print(f"Total unique candidate ChEMBL IDs before name resolution: {len(all_candidate_ids)}")


Indication search: 217 candidate ChEMBL IDs
Target search: 108 candidate ChEMBL IDs
Total unique candidate ChEMBL IDs before name resolution: 299


In [ ]:
def resolve_drug_names(chembl_ids: list[str], batch_size: int = 25) -> dict[str, dict]:
    """Resolve ChEMBL IDs to {chembl_id: {'name': ..., 'max_phase': ...}}, batched via molecule_chembl_id__in.
    Only molecules with a resolvable preferred name are kept (unnamed/experimental compounds are dropped —
    they wouldn't be useful or recognizable in a clinician-facing answer anyway)."""
    resolved = {}
    for i in range(0, len(chembl_ids), batch_size):
        batch = chembl_ids[i:i + batch_size]
        ids_param = ",".join(batch)
        url = f"{CHEMBL_BASE}/molecule.json?molecule_chembl_id__in={ids_param}&limit=1000"
        try:
            resp = requests.get(url, timeout=20)
            resp.raise_for_status()
            for m in resp.json().get("molecules", []):
                pref_name = m.get("pref_name")
                if pref_name:
                    resolved[m["molecule_chembl_id"]] = {
                        "name": pref_name,
                        "max_phase": m.get("max_phase"),
                    }
        except Exception as e:
            print(f"[ChEMBL] Error resolving names for batch starting at {i}: {e}")
        time.sleep(0.3)
    return resolved


name_map = resolve_drug_names(all_candidate_ids)

# Prefer drugs that reached later clinical phases (more established / more likely to have
# PubChem+ChEMBL data worth showing), then cap to MAX_DRUGS.
drug_candidates = sorted(
    ({"chembl_id": cid, **info} for cid, info in name_map.items()),
    key=lambda d: (d.get("max_phase") or 0),
    reverse=True,
)[:MAX_DRUGS]

print(f"\nDiscovered {len(name_map)} named drugs, ingesting top {len(drug_candidates)} (MAX_DRUGS={MAX_DRUGS}):")
for d in drug_candidates:
    print(f"  {d['chembl_id']} — {d['name']} (max_phase={d.get('max_phase')})")



Discovered 299 named drugs, ingesting top 100 (MAX_DRUGS=100):
  CHEMBL468 — THALIDOMIDE (max_phase=4.0)
  CHEMBL554 — LAPATINIB (max_phase=4.0)
  CHEMBL1373 — MODAFINIL (max_phase=4.0)
  CHEMBL1421 — DASATINIB ANHYDROUS (max_phase=4.0)
  CHEMBL553025 — VINORELBINE (max_phase=4.0)
  CHEMBL1201010 — FLUDROCORTISONE ACETATE (max_phase=4.0)
  CHEMBL4204794 — AVAPRITINIB (max_phase=4.0)
  CHEMBL4297723 — CEMIPLIMAB (max_phase=4.0)
  CHEMBL467 — HYDROXYUREA (max_phase=4.0)
  CHEMBL513 — CARMUSTINE (max_phase=4.0)
  CHEMBL803 — CYTARABINE (max_phase=4.0)
  CHEMBL414804 — OXALIPLATIN (max_phase=4.0)
  CHEMBL608533 — MIDOSTAURIN (max_phase=4.0)
  CHEMBL1096882 — FLUDARABINE PHOSPHATE (max_phase=4.0)
  CHEMBL1200485 — SORAFENIB TOSYLATE (max_phase=4.0)
  CHEMBL1201568 — PEGFILGRASTIM (max_phase=4.0)
  CHEMBL1201564 — INTERFERON GAMMA-1B (max_phase=4.0)
  CHEMBL1201529 — TECHNETIUM TC 99M SULFUR COLLOID (max_phase=4.0)
  CHEMBL1201670 — SARGRAMOSTIM (max_phase=4.0)
  CHEMBL1201576 — RITUXIMAB (

## 3. Fetch mechanism-of-action & indication data from ChEMBL

Now querying directly by `chembl_id` (already resolved in step 2), no name-search ambiguity.


In [6]:
def get_chembl_mechanisms(chembl_id: str) -> list[dict]:
    url = f"{CHEMBL_BASE}/mechanism?molecule_chembl_id={chembl_id}&format=json"
    try:
        resp = requests.get(url, timeout=15)
        resp.raise_for_status()
        return resp.json().get("mechanisms", [])
    except Exception as e:
        print(f"[ChEMBL] Error fetching mechanism for '{chembl_id}': {e}")
        return []
    finally:
        time.sleep(0.3)


def get_chembl_indications(chembl_id: str) -> list[dict]:
    url = f"{CHEMBL_BASE}/drug_indication?molecule_chembl_id={chembl_id}&format=json"
    try:
        resp = requests.get(url, timeout=15)
        resp.raise_for_status()
        return resp.json().get("drug_indications", [])
    except Exception as e:
        print(f"[ChEMBL] Error fetching indications for '{chembl_id}': {e}")
        return []
    finally:
        time.sleep(0.3)


chembl_data = {}
for d in drug_candidates:
    chembl_id = d["chembl_id"]
    mechanisms = get_chembl_mechanisms(chembl_id)
    indications = get_chembl_indications(chembl_id)
    chembl_data[chembl_id] = {
        "name": d["name"],
        "max_phase": d.get("max_phase"),
        "mechanisms": [m.get("mechanism_of_action") for m in mechanisms if m.get("mechanism_of_action")],
        "targets": list({m.get("target_chembl_id") for m in mechanisms if m.get("target_chembl_id")}),
        "indications": [i.get("efo_term") for i in indications if i.get("efo_term")][:10],
    }
    print(f"ChEMBL detail: {d['name']} ({chembl_id}) -> "
          f"{len(chembl_data[chembl_id]['mechanisms'])} mechanism(s), "
          f"{len(chembl_data[chembl_id]['indications'])} indication(s)")


ChEMBL detail: THALIDOMIDE (CHEMBL468) -> 1 mechanism(s), 10 indication(s)
ChEMBL detail: LAPATINIB (CHEMBL554) -> 0 mechanism(s), 10 indication(s)
ChEMBL detail: MODAFINIL (CHEMBL1373) -> 1 mechanism(s), 10 indication(s)
ChEMBL detail: DASATINIB ANHYDROUS (CHEMBL1421) -> 6 mechanism(s), 10 indication(s)
ChEMBL detail: VINORELBINE (CHEMBL553025) -> 0 mechanism(s), 10 indication(s)
ChEMBL detail: FLUDROCORTISONE ACETATE (CHEMBL1201010) -> 1 mechanism(s), 10 indication(s)
ChEMBL detail: AVAPRITINIB (CHEMBL4204794) -> 2 mechanism(s), 7 indication(s)
ChEMBL detail: CEMIPLIMAB (CHEMBL4297723) -> 1 mechanism(s), 10 indication(s)
ChEMBL detail: HYDROXYUREA (CHEMBL467) -> 1 mechanism(s), 10 indication(s)
ChEMBL detail: CARMUSTINE (CHEMBL513) -> 3 mechanism(s), 10 indication(s)
ChEMBL detail: CYTARABINE (CHEMBL803) -> 3 mechanism(s), 10 indication(s)
ChEMBL detail: OXALIPLATIN (CHEMBL414804) -> 1 mechanism(s), 10 indication(s)
ChEMBL detail: MIDOSTAURIN (CHEMBL608533) -> 5 mechanism(s), 10 indi

## 4. Fetch drug descriptions from PubChem


In [21]:
def get_pubchem_description(drug_name: str) -> dict | None:
    """Look up a compound by name and return its PubChem CID + text description, or None if not found."""
    # Step 1: resolve name -> CID (PUG-REST)
    cid_url = f"{PUBCHEM_BASE}/compound/name/{drug_name}/cids/JSON"
    try:
        resp = requests.get(cid_url, timeout=15)
        if resp.status_code != 200:
            print(f"[PubChem] No CID found for '{drug_name}' (status {resp.status_code})")
            return None
        cids = resp.json().get("IdentifierList", {}).get("CID", [])
        if not cids:
            print(f"[PubChem] No CID found for '{drug_name}'")
            return None
        cid = cids[0]
    except Exception as e:
        print(f"[PubChem] Error resolving CID for '{drug_name}': {e}")
        return None
    finally:
        time.sleep(0.3)

    # Step 2: fetch the description directly via PUG-REST (simpler than PUG-View)
    desc_url = f"{PUBCHEM_BASE}/compound/cid/{cid}/description/JSON"
    try:
        resp = requests.get(desc_url, timeout=15)
        if resp.status_code != 200:
            print(f"[PubChem] No description found for '{drug_name}' (CID {cid}, status {resp.status_code})")
            return {"cid": cid, "description": None}

        info = resp.json().get("InformationList", {}).get("Information", [])
        # Multiple entries can come back (different curators/sources) — join the ones that have text.
        # Entries without a "Description" field are just CID/title metadata, so skip those.
        descriptions = [entry["Description"] for entry in info if entry.get("Description")]

        return {"cid": cid, "description": " ".join(descriptions) if descriptions else None}
    except Exception as e:
        print(f"[PubChem] Error fetching description for '{drug_name}' (CID {cid}): {e}")
        return {"cid": cid, "description": None}
    finally:
        time.sleep(0.3)


pubchem_data = {}
for d in drug_candidates:
    result = get_pubchem_description(d["name"])
    if result:
        pubchem_data[d["chembl_id"]] = result
    print(f"PubChem: {d['name']} -> {'OK' if result and result.get('description') else 'MISSING/NO DESC'}")


PubChem: THALIDOMIDE -> OK
PubChem: LAPATINIB -> OK
PubChem: MODAFINIL -> OK
PubChem: DASATINIB ANHYDROUS -> OK
PubChem: VINORELBINE -> OK
PubChem: FLUDROCORTISONE ACETATE -> OK
PubChem: AVAPRITINIB -> MISSING/NO DESC
[PubChem] No CID found for 'CEMIPLIMAB' (status 404)
PubChem: CEMIPLIMAB -> MISSING/NO DESC
PubChem: HYDROXYUREA -> OK
PubChem: CARMUSTINE -> OK
PubChem: CYTARABINE -> OK
PubChem: OXALIPLATIN -> MISSING/NO DESC
PubChem: MIDOSTAURIN -> OK
PubChem: FLUDARABINE PHOSPHATE -> OK
PubChem: SORAFENIB TOSYLATE -> OK
[PubChem] No CID found for 'PEGFILGRASTIM' (status 404)
PubChem: PEGFILGRASTIM -> MISSING/NO DESC
[PubChem] No CID found for 'INTERFERON GAMMA-1B' (status 404)
PubChem: INTERFERON GAMMA-1B -> MISSING/NO DESC
PubChem: TECHNETIUM TC 99M SULFUR COLLOID -> MISSING/NO DESC
[PubChem] No CID found for 'SARGRAMOSTIM' (status 404)
PubChem: SARGRAMOSTIM -> MISSING/NO DESC
[PubChem] No CID found for 'RITUXIMAB' (status 404)
PubChem: RITUXIMAB -> MISSING/NO DESC
PubChem: HISTAMINE

In [42]:
len(chembl_data), len(pubchem_data)

(100, 76)

In [23]:
import json
import os

os.makedirs("data", exist_ok=True)

# --- Save ---
with open("data/pubchem_data.json", "w") as f:
    json.dump(pubchem_data, f, indent=2)

with open("data/chembl_data.json", "w") as f:
    json.dump(chembl_data, f, indent=2)

print(f"Saved {len(pubchem_data)} PubChem records and {len(chembl_data)} ChEMBL records to /data")

Saved 76 PubChem records and 100 ChEMBL records to /data


In [ ]:
# # --- Load (run this instead of re-fetching from the APIs) ---
# with open("data/pubchem_data.json") as f:
#     pubchem_data = json.load(f)

# with open("data/chembl_data.json") as f:
#     chembl_data = json.load(f)

# print(f"Loaded {len(pubchem_data)} PubChem records and {len(chembl_data)} ChEMBL records from /data")

In [47]:
import pandas as pd

# --- Step 1: convert each dict of dicts into its own DataFrame ---
chembl_df = pd.DataFrame.from_dict(chembl_data, orient="index").reset_index()
chembl_df = chembl_df.rename(columns={"index": "chembl_id"})

pubchem_df = pd.DataFrame.from_dict(pubchem_data, orient="index").reset_index()
pubchem_df = pubchem_df.rename(columns={"index": "chembl_id"})

print("ChEMBL columns:", chembl_df.columns.tolist())
print("PubChem columns:", pubchem_df.columns.tolist())

# --- Step 2: merge on chembl_id (outer join keeps drugs found in only one source) ---
df = pd.merge(chembl_df, pubchem_df, on="chembl_id", how="outer")
print(f"\nMerged: {len(df)} unique drugs "
      f"({len(chembl_df)} from ChEMBL, {len(pubchem_df)} from PubChem)")

# --- Step 3: convert list-valued columns (mechanisms, targets, indications) to strings ---
list_columns = [col for col in df.columns if df[col].apply(lambda v: isinstance(v, list)).any()]
print("List-type columns found:", list_columns)

for col in list_columns:
    df[col] = df[col].apply(lambda v: "; ".join(v) if isinstance(v, list) else v)

# --- Step 4: completeness score — count populated fields per row ---
FIELDS_TO_CHECK = [c for c in df.columns if c != "chembl_id"]

def field_is_filled(value) -> bool:
    if pd.isna(value):
        return False
    if isinstance(value, str):
        return value.strip() != ""
    return True  # numbers like max_phase (including 0) count as filled

df["completeness_score"] = df[FIELDS_TO_CHECK].apply(
    lambda row: sum(field_is_filled(v) for v in row), axis=1
)

df = df.sort_values("completeness_score", ascending=False).reset_index(drop=True)

print(f"\nCompleteness score range: {df['completeness_score'].min()}–{df['completeness_score'].max()} "
      f"(out of {len(FIELDS_TO_CHECK)})")
df.head(15)

ChEMBL columns: ['chembl_id', 'name', 'max_phase', 'mechanisms', 'targets', 'indications']
PubChem columns: ['chembl_id', 'cid', 'description']

Merged: 100 unique drugs (100 from ChEMBL, 76 from PubChem)
List-type columns found: ['mechanisms', 'targets', 'indications']

Completeness score range: 4–7 (out of 7)


,chembl_id,name,max_phase,mechanisms,targets,indications,cid,description,completeness_score
0,CHEMBL109,VALPROIC ACID,4.0,Succinate semialdehyde dehydrogenase inhibitor,CHEMBL1911,epilepsy; pancreatic carcinoma; sarcoma; HIV i...,3121.0,Valproate (Valproic acid) can cause developmen...,7
1,CHEMBL1096882,FLUDARABINE PHOSPHATE,4.0,Ribonucleoside-diphosphate reductase RR1 inhib...,CHEMBL2095215; CHEMBL2363042,chronic lymphocytic leukemia; lymphoma; acute ...,30751.0,Fludarabine phosphate is a purine arabinonucle...,7
2,CHEMBL112,ACETAMINOPHEN,4.0,Cyclooxygenase inhibitor; Anandamide amidohydr...,CHEMBL4794; CHEMBL2243; CHEMBL2094253,"Fever; osteoarthritis, knee; pain; arthritis; ...",1983.0,Paracetamol is a member of the class of phenol...,7
3,CHEMBL11359,CISPLATIN,4.0,DNA inhibitor,CHEMBL2311221,head and neck squamous cell carcinoma; carcino...,5702198.0,Cisplatin can cause cancer according to an ind...,7
4,CHEMBL1200485,SORAFENIB TOSYLATE,4.0,Serine/threonine-protein kinase RAF inhibitor;...,CHEMBL1974; CHEMBL1936; CHEMBL2041; CHEMBL2095...,hepatocellular carcinoma; neoplasm; small cell...,406563.0,Sorafenib tosylate is an organosulfonate salt....,7
5,CHEMBL1200981,EPIRUBICIN HYDROCHLORIDE,4.0,DNA inhibitor,CHEMBL2311221,adenocarcinoma; hepatocellular carcinoma; acut...,65348.0,Epirubicin hydrochloride is an anthracycline. ...,7
6,CHEMBL1201010,FLUDROCORTISONE ACETATE,4.0,Mineralocorticoid receptor agonist,CHEMBL1994,septic shock; melanoma; sensorineural hearing ...,225609.0,Fludrocortisone acetate is an acetate ester re...,7
7,CHEMBL1201668,NESIRITIDE,4.0,Atrial natriuretic peptide receptor A agonist,CHEMBL1988,hypertension; heart failure; myocardial infarc...,71308561.0,Nesiritide is a polypeptide.,7
8,CHEMBL1282,IMIQUIMOD,4.0,Toll-like receptor 7 agonist,CHEMBL5936,actinic keratosis; basal cell carcinoma; melan...,57469.0,"Imiquimod is an imidazoquinoline fused [4,5-c]...",7
9,CHEMBL1421,DASATINIB ANHYDROUS,4.0,Tyrosine-protein kinase ABL inhibitor; Platele...,CHEMBL2363074; CHEMBL1862; CHEMBL2096618; CHEM...,chronic lymphocytic leukemia; leukemia; multip...,3062316.0,Dasatinib (anhydrous) is an aminopyrimidine th...,7


In [48]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   chembl_id           100 non-null    str    
 1   name                100 non-null    str    
 2   max_phase           100 non-null    str    
 3   mechanisms          100 non-null    str    
 4   targets             100 non-null    str    
 5   indications         100 non-null    str    
 6   cid                 76 non-null     float64
 7   description         66 non-null     str    
 8   completeness_score  100 non-null    int64  
dtypes: float64(1), int64(1), str(7)
memory usage: 67.2 KB


In [ ]:
# df.to_json("data/melanoma_drugs_combined.json", orient="records", indent=2)

# print(f"Saved {len(df)} rows to data/melanoma_drugs_combined.json")

Saved 100 rows to data/melanoma_drugs_combined.json


In [49]:
df.completeness_score.value_counts()

completeness_score
7    57
5    33
6     9
4     1
Name: count, dtype: int64

### Master table

In [50]:
# Only embed drug with all info
all_drugs = df[df.completeness_score == 7]
all_drugs.shape

(57, 9)

## 5. Combine into one record per drug, chunk, embed, and upsert


In [52]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

drug_documents: list[Document] = []

for d in all_drugs.to_dict('records'):
    chembl_id = d["chembl_id"]
    pc = pubchem_data.get(chembl_id, {})
    cb = chembl_data.get(chembl_id, {})
    name = cb.get("name", d["name"])

    if not pc and not cb.get("mechanisms") and not cb.get("indications"):
        print(f"Skipping {name} ({chembl_id}) — no usable data from either source")
        continue

    card = f"""Drug: {name}
ChEMBL ID: {chembl_id}
PubChem CID: {pc.get('cid', 'Unknown')}
Description: {pc.get('description') or 'Not available'}
ChEMBL max clinical phase: {cb.get('max_phase', 'Unknown')}
Mechanism(s) of action: {'; '.join(cb.get('mechanisms', [])) or 'Not available'}
Target ChEMBL ID(s): {', '.join(cb.get('targets', [])) or 'Not available'}
Indications: {', '.join(cb.get('indications', [])) or 'Not available'}
"""

    chunks = splitter.split_text(card)
    for idx, chunk in enumerate(chunks):
        drug_documents.append(
            Document(
                page_content=chunk,
                metadata={
                    "drug_name": name,
                    "pubchem_cid": pc.get("cid"),
                    "chembl_id": chembl_id,
                    "chunk_index": idx,
                    "source": "drug",
                },
            )
        )

print(f"\nTotal drug chunks to embed: {len(drug_documents)}")



Total drug chunks to embed: 113


In [54]:
from pinecone import Pinecone
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

pc_client = Pinecone(api_key=PINECONE_API_KEY)
index = pc_client.Index(PINECONE_INDEX_NAME)

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL, openai_api_key=OPENAI_API_KEY)

drugs_vectorstore = PineconeVectorStore(
    index=index, embedding=embeddings, text_key="text", namespace=DRUGS_NAMESPACE
)

if drug_documents:
    # chembl_id-based IDs: re-running this notebook overwrites existing records rather than duplicating.
    ids = [f"{d.metadata['chembl_id']}-{d.metadata['chunk_index']}" for d in drug_documents]
    drugs_vectorstore.add_documents(documents=drug_documents, ids=ids)
    print(f"Upserted {len(drug_documents)} drug chunks into namespace '{DRUGS_NAMESPACE}'.")
else:
    print("No drug documents to upsert — check discovery/PubChem/ChEMBL results above.")


Upserted 113 drug chunks into namespace 'drugs'.


## 6. Router — decide which namespace(s) to query per question


In [55]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

class RouteDecision(BaseModel):
    query_literature: bool = Field(description="True if the question needs published melanoma research/clinical literature.")
    query_drugs: bool = Field(description="True if the question needs drug-specific data (mechanism, structure, targets, indications).")

router_llm = ChatOpenAI(model=LLM_MODEL, temperature=0, openai_api_key=OPENAI_API_KEY).with_structured_output(RouteDecision)

def route_question(question: str) -> RouteDecision:
    return router_llm.invoke(
        f"Classify what data sources this melanoma research question needs: '{question}'"
    )

# quick check
print(route_question("What is the mechanism of action of vemurafenib?"))
print(route_question("What does the literature say about survival rates for stage IV melanoma?"))
print(route_question("Compare the mechanism of dabrafenib with recent trial outcomes combining it with trametinib"))


query_literature=False query_drugs=True
query_literature=True query_drugs=False
query_literature=True query_drugs=True


## 7. Retrievers for each namespace

In [56]:
literature_vectorstore = PineconeVectorStore(
    index=index, embedding=embeddings, text_key="text", namespace=LITERATURE_NAMESPACE
)

literature_retriever = literature_vectorstore.as_retriever(search_kwargs={"k": TOP_K})
drugs_retriever = drugs_vectorstore.as_retriever(search_kwargs={"k": TOP_K})


## 8. Combined, routed context builder


In [57]:
def format_doc(doc, source_type: str) -> str:
    meta = doc.metadata
    if source_type == "literature":
        header = f"[LITERATURE | PMID: {meta.get('pmid')}] {meta.get('title')} ({meta.get('year')}, {meta.get('journal')})"
    else:
        header = f"[DRUG | {meta.get('drug_name', '').title()} | ChEMBL: {meta.get('chembl_id')} | PubChem CID: {meta.get('pubchem_cid')}]"
    return f"{header}\n{doc.page_content}"


def retrieve_combined(question: str):
    route = route_question(question)
    # fail-safe: if the router is unsure/returns both False, default to literature
    if not route.query_literature and not route.query_drugs:
        route.query_literature = True

    lit_docs, drug_docs = [], []
    if route.query_literature:
        lit_docs = literature_retriever.invoke(question)
    if route.query_drugs:
        drug_docs = drugs_retriever.invoke(question)

    context = "\n\n---\n\n".join(
        [format_doc(d, "literature") for d in lit_docs] + [format_doc(d, "drug") for d in drug_docs]
    )
    return context, lit_docs, drug_docs


def format_sources(lit_docs, drug_docs) -> list[dict]:
    sources = []
    seen_pmids, seen_drugs = set(), set()
    for d in lit_docs:
        pmid = d.metadata.get("pmid")
        if pmid and pmid not in seen_pmids:
            seen_pmids.add(pmid)
            sources.append({
                "type": "literature", "label": f"PMID {pmid}: {d.metadata.get('title')}",
                "url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
            })
    for d in drug_docs:
        name = d.metadata.get("drug_name")
        if name and name not in seen_drugs:
            seen_drugs.add(name)
            cid = d.metadata.get("pubchem_cid")
            sources.append({
                "type": "drug", "label": f"Drug: {name.title()} (ChEMBL {d.metadata.get('chembl_id')})",
                "url": f"https://pubchem.ncbi.nlm.nih.gov/compound/{cid}" if cid else None,
            })
    return sources


## 9. Prompt with a memory slot

`MessagesPlaceholder("chat_history")` lets prior turns flow into the prompt, so follow-up
questions resolve correctly.


In [58]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

SYSTEM_PROMPT = """You are a melanoma research assistant for clinicians and researchers.

Rules you must follow:
- Only answer questions related to melanoma (research literature, biology, and melanoma-relevant drugs).
  If a question is unrelated to melanoma, say this assistant is scoped to melanoma research only.
- Base your answer ONLY on the provided context below plus the conversation history. Do not use outside knowledge.
- Cite literature claims as (PMID: xxxxx) and drug claims as (Drug: <name>, ChEMBL: <id>).
- If the context does not contain enough information, say so explicitly instead of guessing.
- This is a research/literature summary tool, not medical advice — do not give definitive treatment recommendations.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])


## 10. Build the memory-aware, routed RAG chain

In [59]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

llm = ChatOpenAI(model=LLM_MODEL, temperature=0, openai_api_key=OPENAI_API_KEY)

chain = prompt | llm | StrOutputParser()

def to_lc_history(turns: list[dict], max_turns: int = 6) -> list:
    """Convert a list of {'role': ..., 'content': ...} dicts into LC message objects,
    keeping only the most recent `max_turns` exchanges to bound token/cost growth."""
    history = []
    for m in turns[-max_turns * 2:]:
        if m["role"] == "user":
            history.append(HumanMessage(content=m["content"]))
        elif m["role"] == "assistant":
            history.append(AIMessage(content=m["content"]))
    return history


## 11. `answer_question()` helper with explicit conversation memory



In [60]:
def answer_question(question: str, conversation: list[dict]) -> dict:
    """conversation: list of {'role': 'user'|'assistant', 'content': str} dicts (memory, owned by caller)."""
    context, lit_docs, drug_docs = retrieve_combined(question)
    if not lit_docs and not drug_docs:
        return {
            "answer": "No relevant melanoma literature or drug data was found for this question.",
            "sources": [],
        }

    chat_history = to_lc_history(conversation)
    answer_text = chain.invoke({
        "context": context,
        "question": question,
        "chat_history": chat_history,
    })
    return {"answer": answer_text, "sources": format_sources(lit_docs, drug_docs)}


## 12. Test a short multi-turn conversation

In [61]:
conversation = []  # simulates st.session_state.messages in the Streamlit app

def ask(q):
    result = answer_question(q, conversation)
    print(f"Q: {q}\nA: {result['answer']}\n")
    for s in result["sources"]:
        print(f"  [{s['type']}] {s['label']} -> {s['url']}")
    print("\n" + "=" * 80 + "\n")
    conversation.append({"role": "user", "content": q})
    conversation.append({"role": "assistant", "content": result["answer"]})

ask("What is the mechanism of action of dabrafenib?")
ask("What about when it's combined with trametinib — any literature on outcomes?")  # follow-up relies on memory


Q: What is the mechanism of action of dabrafenib?
A: The mechanism of action of dabrafenib is as a serine/threonine-protein kinase B-raf inhibitor (PMID: xxxxx).

  [drug] Drug: Dabrafenib Mesylate (ChEMBL CHEMBL2105729) -> https://pubchem.ncbi.nlm.nih.gov/compound/44516822.0
  [drug] Drug: Vemurafenib (ChEMBL CHEMBL1229517) -> https://pubchem.ncbi.nlm.nih.gov/compound/42611257.0
  [drug] Drug: Trametinib (ChEMBL CHEMBL2103875) -> https://pubchem.ncbi.nlm.nih.gov/compound/11707110.0
  [drug] Drug: Trametinib Dimethyl Sulfoxide (ChEMBL CHEMBL2105741) -> https://pubchem.ncbi.nlm.nih.gov/compound/50992434.0


Q: What about when it's combined with trametinib — any literature on outcomes?
A: Yes, there is literature on the outcomes of combining dabrafenib with trametinib. The COMBI-AD trial demonstrated that adjuvant therapy with dabrafenib and trametinib improved relapse-free survival (RFS) compared to placebo in patients with stage III melanoma (PMID: 32403192.0). The hazard ratio for RFS

In [62]:
ask("What is the mechanism of action of acetaminophen?")

Q: What is the mechanism of action of acetaminophen?
A: The mechanism of action of acetaminophen includes being a cyclooxygenase inhibitor, an anandamide amidohydrolase inhibitor, and a vanilloid receptor opener (Drug: Acetaminophen, ChEMBL: CHEMBL112).

  [drug] Drug: Acetaminophen (ChEMBL CHEMBL112) -> https://pubchem.ncbi.nlm.nih.gov/compound/1983.0
  [drug] Drug: Aspirin (ChEMBL CHEMBL25) -> https://pubchem.ncbi.nlm.nih.gov/compound/2244.0
  [drug] Drug: Indomethacin (ChEMBL CHEMBL6) -> https://pubchem.ncbi.nlm.nih.gov/compound/3715.0


